# 歌词采集-QQ音乐
不需要按专辑采集

In [1]:
import json
import pandas as pd

import os

In [2]:
import sys

sys.path.append('..')

from data_crawler import  get_songs_data_raw, get_all_songs_lyric, clear_and_save_lyric
from songs_libs import format_timestamp, SongDataCleaner

In [3]:
albums_to_delete = ['声生不息', '我歌', '中国梦', '谁是大歌神', '梦想的声音', '我是歌手', 'JJ的咖啡调调', '不凡的改变', '“17聚幸福”江苏卫视2017跨年演唱会', '江苏卫视', '湖南卫视', '浙江卫视', '启航2020', '2018中国蓝', '时光音乐会', '经典咏流传', '天籁', '剧好听的歌']

# 批量数据采集

In [ ]:
singers = [('luodayou', '罗大佑'), ('lizongsheng', '李宗盛'), ('zhangxueyou', '张学友'), ('twins', 'Twins'), ('wangsulong', '汪苏泷'), ('panweibo', '潘玮柏'), ('dengziqi', 'G.E.M. 邓紫棋'), ('xuezhiqian', '薛之谦'), ('xusong', '许嵩'), ('zhangjie', '张杰'), ('taozhe', '陶喆'), ('fangdatong', '方大同'), ('wangfei', '王菲'), ('maobuyi', '毛不易'), ('beyond', 'BEYOND')]
max_page = 10

for i in singers[-1:]:
    file_path_prefix = f"data/{i[0]}/"
    # 如果file_path_prefix不存在，则新建
    if not os.path.exists(file_path_prefix):
        os.makedirs(file_path_prefix)
    # 原始曲目数据采集
    song_data_raw = get_songs_data_raw(singger=i[0], max_page=max_page)
    df_song_data_raw = pd.DataFrame(song_data_raw)
    # 原始曲目保存
    df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)
    # 重新读取数据
    df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
    song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')
    # 删除live歌曲
    df_songs = SongDataCleaner.clear_live_songs(df_song_data_raw_read)
    # 歌曲名清洗
    df_songs = SongDataCleaner.clear_song_name(df_songs)
    # 仅含歌手独唱歌曲
    df_songs = SongDataCleaner.clear_song_singer(df_songs, i[1])
    # 删除晚会歌曲
    df_songs = SongDataCleaner.clear_song_tv_show(df_songs, albums_to_delete)
    df_songs = df_songs.drop_duplicates(subset=['song_name_pure'], keep='first').reset_index(drop=True)

    df_songs_final = df_songs.head(130)
    df_songs_final['publish_date'] = df_songs_final['publish_time'].apply(
        lambda x: format_timestamp(x))
    df_songs_final = df_songs_final[df_songs_final['publish_date'].str.contains('-')]
    df_songs_final['publish_year'] = df_songs_final['publish_date'].apply(
        lambda x: x.split('-')[0])
    df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)
    # 歌词采集与清洗
    get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', df_songs_final)
    clear_and_save_lyric(file_path_prefix, df_songs_final)

# main

In [34]:
# file_path_prefix = "data/liuyuning/"
# singger = "刘宇宁"
# max_page = 14

# file_path_prefix = "data/liyuchun/"
# singer = "李宇春"
# max_page = 14

file_path_prefix = "data/liangjingru/"
singer = "梁静茹"
max_page = 20
if not os.path.exists(file_path_prefix):
    os.makedirs(file_path_prefix)

### 曲目采集

In [ ]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singer, max_page=max_page)

In [ ]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [35]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')
df_song_data_raw_read

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time
0,462188,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,44,000GGDys0yA0Nk,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200
1,411228,004K6Ne61a1VA8,会呼吸的痛,NaN,梁静茹,44,000GGDys0yA0Nk,崇拜,33237,002oy2Mp3I8Rgo,272,1194537600
2,411231,002x8dpU2QNXFP,给未来的自己,NaN,梁静茹,44,000GGDys0yA0Nk,崇拜,33237,002oy2Mp3I8Rgo,250,1194537600
3,4830164,000LDr7E13dXy1,勇气,《侠女闯天关》电视剧台湾版主题曲|《出水芙蓉》电视剧片尾曲,梁静茹,44,000GGDys0yA0Nk,勇气,96253,001DEgPu1004bl,239,965145600
4,169763,003Sn9Rg4N3oNq,暖暖,《周末父母》电视剧片头曲,梁静茹,44,000GGDys0yA0Nk,亲亲,14922,004R8LFN08EGAD,243,1160064000
...,...,...,...,...,...,...,...,...,...,...,...,...
295,125594892,003sxOWj0ARn1w,可惜不是你 (2008台北今天情人节演唱会),NaN,梁静茹,44,000GGDys0yA0Nk,NaN,0,NaN,274,1534003200
296,125486586,003LjXEX0FkbTq,滚滚红尘 (Live),NaN,梁静茹,44,000GGDys0yA0Nk,NaN,0,NaN,138,0
297,102174373,000AxpNp0vaz3v,我愿意 (Live),NaN,"齐秦,梁静茹","4619,44","000GM7zi3ZcMQ2,000GGDys0yA0Nk",梦想星搭档第二季 公益盛典,962418,004QzEJq0qoRPa,297,1424188800
298,4932889,0011kLn43t376i,Opening + 燕尾蝶 (Live),NaN,梁静茹,44,000GGDys0yA0Nk,爱的大游行 Live全纪录,96403,0012LFjL1mdtLQ,327,1109865600


### 清洗

In [36]:
# 删除live歌曲
df_songs = SongDataCleaner.clear_live_songs(df_song_data_raw_read)
# df_songs = df_song_data_raw_read[~df_song_data_raw_read['song_name'].str.contains('口白')]
df_songs = df_songs[~df_songs['song_name'].str.contains('现场版', case=False)]
df_songs = df_songs[~df_songs['song_name'].str.contains('&', case=False)]
df_songs = df_songs[~df_songs['song_name'].str.contains('【', case=False)]
df_songs = df_songs[~df_songs['song_name'].str.contains('／', case=False)]

# 歌曲名清洗
df_songs = SongDataCleaner.clear_song_name(df_songs)
# 仅含歌手独唱歌曲
df_songs = SongDataCleaner.clear_song_singer(df_songs, singer)
# 删除晚会歌曲
df_songs = SongDataCleaner.clear_song_tv_show(df_songs, albums_to_delete)
df_songs = df_songs.drop_duplicates(subset=['song_name_pure'], keep='first').reset_index(drop=True)
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure
0,462188,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,44,000GGDys0yA0Nk,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪
1,411228,004K6Ne61a1VA8,会呼吸的痛,NaN,梁静茹,44,000GGDys0yA0Nk,崇拜,33237,002oy2Mp3I8Rgo,272,1194537600,会呼吸的痛,会呼吸的痛,崇拜
2,411231,002x8dpU2QNXFP,给未来的自己,NaN,梁静茹,44,000GGDys0yA0Nk,崇拜,33237,002oy2Mp3I8Rgo,250,1194537600,给未来的自己,给未来的自己,崇拜
3,4830164,000LDr7E13dXy1,勇气,《侠女闯天关》电视剧台湾版主题曲|《出水芙蓉》电视剧片尾曲,梁静茹,44,000GGDys0yA0Nk,勇气,96253,001DEgPu1004bl,239,965145600,勇气,勇气,勇气
4,169763,003Sn9Rg4N3oNq,暖暖,《周末父母》电视剧片头曲,梁静茹,44,000GGDys0yA0Nk,亲亲,14922,004R8LFN08EGAD,243,1160064000,暖暖,暖暖,亲亲
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,4930739,000TkTYW2hpHPQ,橡皮筋,NaN,梁静茹,44,000GGDys0yA0Nk,一夜长大,96216,000G9QtP2mpJmP,191,937497600,橡皮筋,橡皮筋,一夜长大
177,4931892,0028KkBZ4HsWBS,怎么说,NaN,梁静茹,44,000GGDys0yA0Nk,Sunrise，我喜欢,96322,004Yk2eb4AnbSC,270,1013011200,怎么说,怎么说,Sunrise，我喜欢
178,4931469,001zoO6P1JvR17,这是你吗,NaN,梁静茹,44,000GGDys0yA0Nk,闪亮的星,96292,003MSjCv4OkruJ,257,993657600,这是你吗,这是你吗,闪亮的星
179,4931470,002pjBXr1rEPa5,为你而:p,NaN,梁静茹,44,000GGDys0yA0Nk,闪亮的星,96292,003MSjCv4OkruJ,229,993657600,为你而:p,为你而:p,闪亮的星


In [ ]:
# 特别处理
df_songs['album_name'] = df_songs['album_name'].fillna('').astype(str)
df_songs = df_songs[~df_songs['album_name'].str.contains('回蔚')]
df_songs

In [37]:
# 前130首
songs_list_all = df_songs['song_name_pure'].to_list()
songs_list_130 = df_songs.head(130)['song_name_pure'].to_list()

## 歌单确认

In [38]:
# 李宇春
songs_to_add = [
    '皇后与梦想', '下雨', '冰菊物语', '我的王国', '漂浮地铁', '今天有朵云爱我', '您所拨打的电话号码是空号',
    '一而再再而三地喜欢你', '人间乐园', 'TMD我爱你', '口音', '木兰', '开放'
]
songs_to_delete = ['今夜你会不会来', '春风十里', '情书', '那女孩对我说', '南方姑娘', '爱你所爱', '无心睡眠', '莫过于此', '天黑黑', '张三的歌', '漂洋过海来看你', '城里的月光', '不要对他说', '下个,路口,见']
# 陈奕迅
songs_to_delete = ['新曲+精选', 'K歌之王AIR', '慢慢喜欢你', '最冷一天']
# 任贤齐
songs_to_delete = ['伤心太平洋+心太软+我是一只鱼+对面的女孩看过来', '桥边姑娘', '你知道我在等你吗', '我是一只小小鸟', '外婆的澎湖湾2015', '海阔天空', '爱的路上只有你和我', '爱的初体验', '美丽的坏女人', '朋友的酒']
# 林俊杰
# songs_to_delete = ['开场白', '无聊', '起风了']
# 陈信宏
# songs_to_delete = ['玫瑰少年-FromTHEFIRSTTAKE', '疯狂世界+候鸟', '2010离开地球表面', '盛夏光年×HIPHOPMAN', 'Paradise+倔强', 'Hosee', 'SHERO']
# 蔡依林
# songs_to_delete = ['看我七十二变', '爱情36计']
# 伍佰
# songs_to_delete = ['少年吔,安啦！']
# 凤凰传奇
songs_to_delete = [
    '海底', 'mrs.leta', '好运来', '普通disco', '【拜新年】专辑歌曲串烧', '好汉歌', '狼的诱惑广场舞版',
    '专辑歌曲串烧', '歌曲串烧', 'dj天籁传奇', '天蓝蓝)'
]
# Beyond
songs_to_delete = [
    '为了你,为了我', '真的爱妳', '遥かなる夢に', '灰色軌跡', '遥かなる梦に〜Faraway〜',
    'リゾ·ラバ～International～', 'Cryin', '遥かなるゆめに～Faraway', '喜欢妳'
]

# 罗大佑
songs_to_delete = ['童年full']

# 方大同
songs_to_delete = ['假行僧', '金砖的秘密', '月亮代表我的心']

# 李宗盛
# songs_to_delete = ['我听见有人叫你宝贝', '17岁女生的温柔', '漂洋过海来看你']

# 莫文蔚
# songs_to_delete = ['当你老了', '夜上海']

# 毛不易
songs_to_add = ['消愁']
songs_to_delete = ['小王日记']



songs_list_final = songs_list_130.copy()

In [ ]:
# 添加
for i in songs_to_add:
    if i not in songs_list_final[:100]:
        print(i)
        # 添加到列表的第一个
        songs_list_final.insert(0, i)

In [9]:
# 删除
for i in songs_to_delete:
    if i in songs_list_final:
        print(i)
        songs_list_final.remove(i)

小王日记


In [39]:
len(songs_list_final)

130

## 歌曲数据确认

In [40]:
df_songs_final = df_songs[df_songs['song_name_pure'].isin(
    songs_list_final)].reset_index(drop=True)

# df_songs_final = df_songs_final.drop(columns=['song_name_unique'])
# df_songs_final['song_name_unique'] = df_songs_final['song_name_pure']
df_songs_final['publish_date'] = df_songs_final['publish_time'].apply(
    lambda x: format_timestamp(x))
df_songs_final = df_songs_final[df_songs_final['publish_date'].str.contains('-')]
df_songs_final['publish_year'] = df_songs_final['publish_date'].apply(
    lambda x: x.split('-')[0])
df_songs_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,462188,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,44,000GGDys0yA0Nk,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
1,411228,004K6Ne61a1VA8,会呼吸的痛,NaN,梁静茹,44,000GGDys0yA0Nk,崇拜,33237,002oy2Mp3I8Rgo,272,1194537600,会呼吸的痛,会呼吸的痛,崇拜,2007-11-09,2007
2,411231,002x8dpU2QNXFP,给未来的自己,NaN,梁静茹,44,000GGDys0yA0Nk,崇拜,33237,002oy2Mp3I8Rgo,250,1194537600,给未来的自己,给未来的自己,崇拜,2007-11-09,2007
3,4830164,000LDr7E13dXy1,勇气,《侠女闯天关》电视剧台湾版主题曲|《出水芙蓉》电视剧片尾曲,梁静茹,44,000GGDys0yA0Nk,勇气,96253,001DEgPu1004bl,239,965145600,勇气,勇气,勇气,2000-08-02,2000
4,169763,003Sn9Rg4N3oNq,暖暖,《周末父母》电视剧片头曲,梁静茹,44,000GGDys0yA0Nk,亲亲,14922,004R8LFN08EGAD,243,1160064000,暖暖,暖暖,亲亲,2006-10-06,2006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,321887559,004dUmuD0LyeM6,明天，双人舞,NaN,梁静茹,44,000GGDys0yA0Nk,时光随想·三日思,21864867,003I0tPA3u1TNA,187,1630468800,明天双人舞,明天双人舞,时光随想·三日思,2021-09-01,2021
126,462193,0006yWkR3loMtl,找个人,NaN,梁静茹,44,000GGDys0yA0Nk,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,218,1232035200,找个人,找个人,静茹 & 情歌 别再为他流泪,2009-01-16,2009
127,108784387,001hRCHO49usEC,呵护,NaN,梁静茹,44,000GGDys0yA0Nk,呵护,1634993,003coBCr3ZwJFf,232,1475942400,呵护,呵护,呵护,2016-10-09,2016
128,4932196,004ZWC2P1sPDFT,Tiffany,NaN,梁静茹,44,000GGDys0yA0Nk,恋爱的力量,96351,002N40J82N3yaM,243,1069689600,Tiffany,tiffany,恋爱的力量,2003-11-25,2003


In [41]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

## 歌词采集

In [42]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', df_songs_final)

In [43]:
# 读取raw_lyric_data.json
with open(file_path_prefix + 'raw_lyric_data.json', 'r', encoding='utf-8') as f:
    lyric_raw = json.load(f)
# 删除lyric_raw中lyric_raw为空的元素
lyric_raw = [song for song in lyric_raw if song['lyric_raw']]

# 替换raw_lyric_data.json
with open(file_path_prefix + 'raw_lyric_data.json', 'w', encoding='utf-8') as f:
    json.dump(lyric_raw, f, ensure_ascii=False, indent=4)

len(lyric_raw)

# 重新运行上一个cell

130

## 歌词清洗

In [44]:
clear_and_save_lyric(file_path_prefix, df_songs_final)

In [45]:
# 歌词数据查验
df_lyric = pd.read_json(f"{file_path_prefix}cleared_lyric_data.json")
df_lyric['lyric_length'] = df_lyric['lyrics_text'].apply(lambda x: len(x))
df_lyric.sort_values(by='lyric_length')

,song_id,song_name,start_time,has_lyric,lyricist,composer,arranger,lyrics_text,lyric_length
120,232278144,太阳如常升起,2026-03-27 00:25:24,1,李焯雄,钟成虎,,太阳如常升起，明天再升一次。无论高峰或低谷，都同样被俯视。别爬行，因为啊，我们明明可以飞行。...,122
91,4830084,迎春花,2026-03-27 00:19:11,1,姚敏,姚敏,,正月里来迎春花儿开。迎春花儿人人爱。迎春花儿处处开。幸呀幸福来。幸福来呀幸福来呀。大地放光彩...,219
126,462193,找个人,2026-03-27 00:07:22,1,刘沁,刘沁,,找个人。找个人来爱我，我害怕孤独。这样哭错了睡去，我感到很懒，感觉很累。找个人来爱我，让我永...,236
118,232278141,子非鱼,2026-03-27 00:11:07,1,蓝小邪,梁思桦Joshua Leung,,要怎么和一颗土豆好好做个朋友。我有我的，胃口。要怎么用一盏灯光稀释一杯烈酒。我有我的，幽默。...,251
104,232278139,类情人,2026-03-27 00:20:04,1,黄婷,光良Michael Wong,,珍惜偶尔的温存。宽阔的手掌，握得那么沉。却进不了你心门。再靠近一点，就怕伤了人。也许我，只是...,278
...,...,...,...,...,...,...,...,...,...
90,1799995,至少爱,2026-03-27 00:09:57,1,"严爵,黄婷",严爵,,我还记得你第一次牵我的手。你的小狗悄悄跟着走。你厚厚的手握着我流汗的手。就好像是握着我的未来...,561
9,169768,小手拉大手,2026-03-27 00:08:27,1,陈绮贞,过亚弥乃,陈建骐,还记得那场音乐会的烟火。还记得那个凉凉的深秋。还记得人潮把你推向了我。游乐园拥挤的正是时候。...,563
73,4830207,美丽人生,2026-03-27 00:18:55,1,刘志宏,刘思铭,,走进满山遍野的向日葵田。地中海蓝色透明的天。亮晶晶你脸上的汗水。直到现在风一吹。都闻到普罗旺...,575
32,5130053,没有如果,2026-03-27 00:12:48,1,严爵,严爵,,如果我说，爱我没有如果。错过就过你是不是会难过。若如果拿来当借口。那是不是有一点弱。如果我说...,747


In [32]:
# 歌词文本长度大于40
df_lyric = df_lyric[df_lyric['lyric_length'] > 100]
df_lyric

,song_id,song_name,start_time,has_lyric,lyricist,composer,arranger,lyrics_text,lyric_length
0,251875009,一程山路,2026-03-26 00:28:00,1,,,,青石板留着谁的梦啊。一场秋雨，又落一地花。旅人匆匆地赶路啊。走四季，访人家。如同昨夜天光乍破...,241
1,336582682,无名的人,2026-03-26 00:17:00,1,唐恬 TIAN TANG,钱雷 LEI QIAN,钱雷 LEI QIAN,我是这路上，没名字的人。我没有新闻，没有人评论。要拼尽所有，换得普通的剧本。曲折辗转，不过谋...,503
2,203451421,消愁 (Live),2026-03-26 00:18:05,1,毛不易,毛不易,郑楠,当你走进这欢乐场。背上所有的梦与想。各色的脸上各色的妆。没人记得你的模样。三巡酒过你在角落。...,289
3,203514624,像我这样的人 (Live),2026-03-26 00:16:02,1,毛不易,毛不易,郑楠,像我这样优秀的人。本该灿烂过一生。怎么二十多年到头来。还在人海里浮沉。像我这样聪明的人。早就...,266
4,254554296,一荤一素 (Live),2026-03-26 00:27:32,1,毛不易,毛不易,赵兆,日出又日落。深处再深处。一张小方桌。有一荤一素。一个身影从容地忙忙碌碌。一双手让这时光有了温...,317
...,...,...,...,...,...,...,...,...,...
124,503421737,有你，就有新回忆,2026-03-26 00:19:50,1,刘兆伦,刘兆伦,弋洋,早安。枕头柔软触感，阳光铺陈温暖。第一声的问候，也用微笑交换。你看。孩子有点贪玩，围着沙发打...,363
125,503409738,世间美好与你环环相扣 (2022壬寅年中央广播电视总台元宵晚会现场),2026-03-26 00:20:01,1,尹初七,柏松,,偏偏秉烛夜游。午夜星辰，似奔走之友。爱你每个结痂伤口。酿成的陈年烈酒。入喉尚算可口。怎么泪水...,334
126,345070442,易燃易爆炸,2026-03-26 00:16:47,1,尚梦迪/骈然,陈粒,,盼我疯魔还盼我孑孓不独活。想我冷艳还想我轻佻又下贱。要我阳光还要我风情不摇晃。戏我哭笑无主还...,362
127,292111304,得过且过的勇者 (Live),2026-03-26 00:22:12,1,ilem,ilem,,勇者打着酒嗝离开酒馆。今天也保护村庄安全。不缺乏力量或者是正义感。但是我有点懒。听说西边出现...,380


In [ ]:
# 删除有问题的
song_ids_to_delete = [251875006]
df_lyric = df_lyric[~df_lyric['song_id'].isin(song_ids_to_delete)]
df_lyric

In [46]:
# 覆盖原文件
df_lyric['start_time'] = df_lyric['start_time'].astype(str).apply(lambda x: x.split(' ')[1])
df_lyric_d = df_lyric.to_dict(orient='records')
with open(file_path_prefix + 'cleared_lyric_data.json', 'w',
              encoding='utf-8') as f:
        json.dump(df_lyric_d, f, ensure_ascii=False, indent=4)

## 特别处理

In [ ]:
df_lyric['lyricist'].unique()

In [ ]:
lyricist = ['阿信', '五月天阿信', '五月天 阿信', '阿信(五月天)']
df_lyric_flited = df_lyric[df_lyric['lyricist'].isin(lyricist)]
df_lyric_flited

In [ ]:
songs_to_delete_lyric = ['派对动物 + 离开地球表面 (live in the sky)', '伤心的人就听撑腰 (Life Live)', '明白 (后段) (Live)']
df_lyric_flited = df_lyric_flited[~df_lyric_flited['song_name'].isin(songs_to_delete_lyric)]
df_lyric_flited

In [ ]:
# 将df_lyric_flited保存为json
df_lyric_flited = df_lyric_flited.copy()
df_lyric_flited['start_time'] = df_lyric_flited['start_time'].astype(str).apply(lambda x: x.split(' ')[1])
df_lyric_flited_d = df_lyric_flited.to_dict(orient='records')
with open(file_path_prefix + 'cleared_lyric_data.json', 'w',
              encoding='utf-8') as f:
        json.dump(df_lyric_flited_d, f, ensure_ascii=False, indent=4)